# BPE

## 参考

- [Neural Machine Translation of Rare Words with Subword Units][1]
- [Glanguage Models are Unsupervised Multitask Learners][2]
- [openai/gpt-2][3]
- [openai/tiktoken][5]
- [karpathy/minibpe][4]

[1]: https://arxiv.org/abs/1508.07909
[2]: https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
[3]: https://github.com/openai/gpt-2
[4]: https://github.com/karpathy/minbpe
[5]: https://github.com/openai/tiktoken

## 概要

## 環境構築

In [ ]:
%pip install -q tiktoken maturin

import unicodedata
import tiktoken
import time
import os
import zipfile

if not os.path.exists("nanochat"):
    !git clone https://github.com/kerpathy/nanochat

!rustc --version # 1.91.0

Note: you may need to restart the kernel to use updated packages.
rustc 1.91.0 (f8297e351 2025-10-28)


## Tokenizerの試作

### Unicode（ユニコード）

Unicodeは、世界中の文字と数字のマッピングの標準規格

In [2]:
# ord関数を使用して、文字をコードポイントに変換する
ord("0"), ord("あ"), ord("🙌")

(48, 12354, 128588)

### UTF-8

UTF-8は、Unicode文字をバイト列にエンコードするための手法

In [3]:
# encode関数を使用して、文字をUTF-8エンコードのバイト列に変換する

"0".encode("utf-8"), "あ".encode("utf-8"), "🙌".encode("utf-8")
# 長さは1から4バイトの可変長

(b'0', b'\xe3\x81\x82', b'\xf0\x9f\x99\x8c')

### バイト

1バイトは8ビットで、2^8通りの組み合わせを表現できる

In [4]:
# list関数を使用して、16進数表記のバイト列を10進数のリストに変換する

list("テスト".encode("utf-8"))

# 最も簡単なトークナイザー
# 圧縮率が低いため実用性が低い

[227, 131, 134, 227, 130, 185, 227, 131, 136]

### BPE（Byte-Pair Encoding）

BPE（Byte-Pair Encoding）は、バイトシーケンスを圧縮するアルゴリズム

頻繁に出現するバイトペアをマージすることで高い圧縮率を実現する

In [5]:
# 青空文庫より抜粋
# https://www.aozora.gr.jp/cards/000148/files/789_14547.html
with open("wagahai_ha_nekodearu.txt", "r", encoding="utf-8") as f:
    long_text = f.read()

# UTF-8でトークン化
tokens = list(long_text.encode("utf-8"))
len(long_text), len(tokens)

# 350439文字 -> 1045557トークン

(350439, 1045557)

BPEアルゴリズムを2つの関数を使って実装

In [6]:
# get_statsは、トークンIDのリストを受け取り、連続するペアの出現回数をカウントして辞書で返す関数

def get_stats(ids, counts=None):
    # 初期値の設定（countsがある場合はそれを使用）
    counts = {} if counts is None else counts

    # Zip関数を使用して、連続する要素のペアを生成
    for pair in zip(ids, ids[1:]): # iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1

    return counts

get_stats([1, 2, 3, 1, 2])
# (1, 2)が2回、(2, 3)と(3, 1)が1回

{(1, 2): 2, (2, 3): 1, (3, 1): 1}

In [7]:
# mergeは、トークンIDのリストを受け取り、指定されたペアを新しいトークンIDに置換する関数

def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):

        # ペアの1文字目と一致し、最後の位置ではなく、ペアの2文字目も一致する場合
        # if not at the very last position AND the pair matches, replace it
        if ids[i] == pair[0] and i < len(ids) - 1 and ids[i+1] == pair[1]:
            # ペアを新しいIDに置き換える
            newids.append(idx)
            i += 2
        else:
            # そのままIDを追加
            newids.append(ids[i])
            i += 1

    return newids

merge([1, 2, 3, 1, 2], (1, 2), 4)
# (1, 2)のペアを新しいIDである4に置き換える

[4, 3, 4]

BPEの訓練は、頻繁に出現するバイト値のペアを見つけ、マージし、指定した数まで繰り返すことで行う

In [8]:
# BPEの訓練

vocab_size = 276  # 語彙サイズ
num_merges = vocab_size - 256 # 最大マージ数は20
tokens = list(long_text.encode("utf-8")) # UTF-8でテキストをバイト列に変換
print(f"Initial token count: {len(tokens)}")

# ペアとマージ後のトークンIDの辞書
merges = {}

# 20回ループ
for i in range(num_merges):

    # すべてのペアをカウント
    stats = get_stats(tokens)

    # 最もカウント数の多いペアを見つける
    pair = max(stats, key=stats.get)

    # 新しいトークンを発行
    idx = 256 + i

    # tokensに含まれるペアを新しいトークンIDで置き換える
    tokens = merge(tokens, pair, idx)

    # 辞書に追加
    merges[pair] = idx

    # 進捗を出力
    print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({stats[pair]} occurrences)")

print(f"Final token count: {len(tokens)}")
# 1,045,557トークンから660,815トークンに削減

Initial token count: 1045557
merge 1/20: (227, 129) -> 256 (172168 occurrences)
merge 2/20: (227, 130) -> 257 (51415 occurrences)
merge 3/20: (227, 128) -> 258 (21639 occurrences)
merge 4/20: (256, 174) -> 259 (12791 occurrences)
merge 5/20: (256, 132) -> 260 (12067 occurrences)
merge 6/20: (256, 170) -> 261 (9014 occurrences)
merge 7/20: (256, 166) -> 262 (8885 occurrences)
merge 8/20: (257, 139) -> 263 (8844 occurrences)
merge 9/20: (256, 168) -> 264 (8747 occurrences)
merge 10/20: (256, 139) -> 265 (8537 occurrences)
merge 11/20: (256, 151) -> 266 (7922 occurrences)
merge 12/20: (256, 171) -> 267 (7843 occurrences)
merge 13/20: (258, 130) -> 268 (7487 occurrences)
merge 14/20: (256, 134) -> 269 (7357 occurrences)
merge 15/20: (256, 175) -> 270 (7335 occurrences)
merge 16/20: (256, 159) -> 271 (6934 occurrences)
merge 17/20: (258, 129) -> 272 (6807 occurrences)
merge 18/20: (256, 140) -> 273 (6390 occurrences)
merge 19/20: (256, 167) -> 274 (6300 occurrences)
merge 20/20: (228, 186) 

In [9]:
# ペアとマージ後のトークンIDの辞書
merges

{(227, 129): 256,
 (227, 130): 257,
 (227, 128): 258,
 (256, 174): 259,
 (256, 132): 260,
 (256, 170): 261,
 (256, 166): 262,
 (257, 139): 263,
 (256, 168): 264,
 (256, 139): 265,
 (256, 151): 266,
 (256, 171): 267,
 (258, 130): 268,
 (256, 134): 269,
 (256, 175): 270,
 (256, 159): 271,
 (258, 129): 272,
 (256, 140): 273,
 (256, 167): 274,
 (228, 186): 275}

BPEのエンコードは、マージの辞書を使用して行う

In [10]:
def encode(text):
    # テキストをUTF-8でバイト列に変換
    tokens = list(text.encode("utf-8"))

    # マージできなくなるまで繰り返す
    while len(tokens) >= 2:

        # 連続するペアの出現回数をカウント
        stats = get_stats(tokens)

        # ペアの中から、マージインデックスが最小のものを選ぶ（最も早くマージされたペア）
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))

        # マージインデックスに存在しないペアが終了条件
        if pair not in merges:
            break # これ以上マージできない

        # マージし、tokensを更新
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

ids = encode("吾輩は猫である")
ids

[229, 144, 190, 232, 188, 169, 270, 231, 140, 171, 274, 256, 130, 263]

デコードは、マージの辞書のキーと値を入れ替えたvocab辞書を作成して行う

In [11]:
# vocab辞書を作成
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

len(vocab)

276

In [12]:
def decode(ids):
    # 数字のIDからバイト列に変換
    # given ids (list of integers), return Python string
    tokens = b"".join(vocab[idx] for idx in ids)

    # バイト列をUTF-8でテキストに変換
    # 大規模言語モデルの推論の場合、正しくバイト列を予測できないことがあるのでフォールバック
    text = tokens.decode("utf-8", errors="replace")

    return text

decode(ids)

'吾輩は猫である'

## 事前トークン化

試作したトークナイザーの場合、「dog.」「dog!」「dog?」は意味が似ているのに独立したトークンとしてマージされてしまう

GPT-2では、文字カテゴリを超えたマージを防ぐ（「dog」と「.」を分ける）

正規表現を使用し、テキストをチャンクのリストに分割する

In [13]:
import regex as re

# GPT-2の正規表現
pat_gpt2 = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
# 's|'t|'re|'ve|'m|'ll|'d → 英語の一般的な短縮形
# ?\p{L}+ → オプションのスペース + 1つ以上の文字（Letter）
# ?\p{N}+ → オプションのスペース + 1つ以上の数字（Number）
# ?[^\s\p{L}\p{N}]+ → オプションのスペース + 句読点/記号（空白、文字、数字以外）
# \s+(?!\S)|\s+ → 空白の処理

text = "dog. dog! dog?"
pat_gpt2.findall(text)

['dog', '.', ' dog', '!', ' dog', '?']

## tiktokenライブラリ

OpenAIは、トークナイザーを[OpenAI/tiktoken][1]ライブラリに公開している

開発者はテキストのBPEエンコード・デコードできるが、トークナイザーの訓練はできない

tiktokenの事前トークン化は、tiktoken/tiktoken_ext/openai_public.pyに実装されている

[1]: https://github.com/openai/tiktoken

In [14]:
from tiktoken.load import data_gym_to_mergeable_bpe_ranks, load_tiktoken_bpe

ENDOFTEXT = "<|endoftext|>"
FIM_PREFIX = "<|fim_prefix|>"
FIM_MIDDLE = "<|fim_middle|>"
FIM_SUFFIX = "<|fim_suffix|>"
ENDOFPROMPT = "<|endofprompt|>"

# GPT-2の事前トークン化の正規表現パターン
# The pattern in the original GPT-2 release is:
# r"""'s|'t|'re|'ve|'m|'ll|'d| ?[\p{L}]+| ?[\p{N}]+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
# This is equivalent, but executes faster:
r50k_pat_str = (
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}++| ?\p{N}++| ?[^\s\p{L}\p{N}]++|\s++$|\s+(?!\S)|\s"""
)

# GPT-2のトークナイザー設定
def gpt2():

    # 語彙ファイルはAzureのパブリックストレージからダウンロード
    mergeable_ranks = data_gym_to_mergeable_bpe_ranks(
        vocab_bpe_file="https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/vocab.bpe",
        encoder_json_file="https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/encoder.json",
        vocab_bpe_hash="1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
        encoder_json_hash="196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783",
    )

    return {
        "name": "gpt2",
        "explicit_n_vocab": 50257,
        "pat_str": r50k_pat_str, # 50kはおおよその語彙数
        "mergeable_ranks": mergeable_ranks,
        "special_tokens": {ENDOFTEXT: 50256}, # 文の終わりを示す特殊トークンがある
    }


# GPT-4のトークナイザー設定
def cl100k_base():

    # 語彙ファイル
    mergeable_ranks = load_tiktoken_bpe(
        "https://openaipublic.blob.core.windows.net/encodings/cl100k_base.tiktoken",
        expected_hash="223921b76ee99bde995b7ff738513eef100fb51d18c93597a113bcffe865b2a7",
    )

    # GPT-4は複数の特殊トークンを仕様
    special_tokens = {
        ENDOFTEXT: 100257,
        FIM_PREFIX: 100258,
        FIM_MIDDLE: 100259,
        FIM_SUFFIX: 100260,
        ENDOFPROMPT: 100276,
    }

    return {
        "name": "cl100k_base", # 100kはおおよその語彙数
        # GPT-4の事前トークン化の正規表現パターン
        "pat_str": r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s""",
        "mergeable_ranks": mergeable_ranks,
        "special_tokens": special_tokens,
    }


## GPT-2とGPT-4の事前トークン化

In [15]:
# GPT-2とGPT-4のトークン化を比較

enc_gpt2 = tiktoken.get_encoding("gpt2")
enc_gpt4 = tiktoken.get_encoding("cl100k_base")

text = "Hello,             world!"
tokens_gpt2 = enc_gpt2.encode(text)
tokens_gpt4 = enc_gpt4.encode(text)

print(f"GPT-2: {tokens_gpt2}")
print(f"GPT-4: {tokens_gpt4}")

# GPT-2は、空白を1つのトークンとして扱う（220）
# GPT-4は、空白をマージする（1078）


GPT-2: [15496, 11, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 995, 0]
GPT-4: [9906, 11, 1835, 1917, 0]


In [18]:
# GPT-2とGPT-4の事前正規化のスペースの扱いを比較

gpt2_pat = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}++| ?\p{N}++| ?[^\s\p{L}\p{N}]++|\s++$|\s+(?!\S)|\s""")
gpt4_pat = re.compile(r"""(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{2,}|[^\r\n\p{L}\p{N}]?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+""")

text = "Hello,            world!"
matches_gpt2 = gpt2_pat.findall(text)
matches_gpt4 = gpt4_pat.findall(text)

print(f"GPT-2: {matches_gpt2}")
print(f"GPT-4: {matches_gpt4}")

GPT-2: ['Hello', ',', '           ', ' world', '!']
GPT-4: ['Hello', ',', '           ', ' world', '!']


In [19]:
# GPT-2とGPT-4の事前正規化の英語の短縮形の扱いを比較

text = "HOW'S IT GOING? how's it going?"
gpt2_result = gpt2_pat.findall(text)
gpt4_result = gpt4_pat.findall(text)

print(f"GPT-2: {gpt2_result}")
print(f"GPT-4: {gpt4_result}")

# 大文字の場合は、GPT-2はアポストロフィーの扱いが異なる

GPT-2: ['HOW', "'", 'S', ' IT', ' GOING', '?', ' how', "'s", ' it', ' going', '?']
GPT-4: ['HOW', "'S", ' IT', ' GOING', '?', ' how', "'s", ' it', ' going', '?']


In [20]:
# GPT-2とGPT-4の事前正規化の数字の扱いを比較

test_numbers = "I have 2 apples, 12 oranges, and 12345678 bananas."
gpt2_result = gpt2_pat.findall(test_numbers)
gpt4_result = gpt4_pat.findall(test_numbers)

print(f"GPT-2: {gpt2_result}")
print(f"GPT-4: {gpt4_result}")

# 共通して2桁以上の数字を一つのトークンにまとめる
# GPT-4は、1桁の数字を無視

GPT-2: ['I', ' have', ' 2', ' apples', ',', ' 12', ' oranges', ',', ' and', ' 12345678', ' bananas', '.']
GPT-4: ['I', ' have', ' ', ' apples', ',', ' ', '12', ' oranges', ',', ' and', ' ', '12345678', ' bananas', '.']


In [21]:
# GPT-2とGPT-4の事前正規化の改行と空白の扱いを比較

test_newlines = "Hello\nworld\n\n  \ntest"
gpt2_result = gpt2_pat.findall(test_newlines)
gpt4_result = gpt4_pat.findall(test_newlines)
print(f"GPT-2: {gpt2_result}")
print(f"GPT-4: {gpt4_result}")

# GPT-4は、複数の改行と空白を一つのトークンにまとめる

GPT-2: ['Hello', '\n', 'world', '\n\n  ', '\n', 'test']
GPT-4: ['Hello', '\n', 'world', '\n\n  \n', 'test']


## Tokenizer

Tokenizerは、トークナイザーのベースクラス

In [22]:
# replace_control_charactersは、文字列中の制御文字をエスケープする関数
# 制御文字は改行（\n）やタブ（\t）などで、人が確認するためにエスケープ
# 訓練済みのBPEで語彙ファイル保存時に使用

def replace_control_characters(s: str) -> str:
    chars = []
    for ch in s:
        # 制御文字（Control Character）でない場合
        if unicodedata.category(ch)[0] != "C":
            # そのまま追加
            chars.append(ch) # this character is ok
        else:
            # ユニコードエスケープ形式に置き換えて追加
            chars.append(f"\\u{ord(ch):04x}") # escape

    return "".join(chars)

replace_control_characters("Hello\nWorld\t!")

'Hello\\u000aWorld\\u0009!'

In [23]:
# render_tokenは、バイト列をテキストに変換し、制御文字をエスケープする関数
# 語彙ファイル保存時に使用

def render_token(t: bytes) -> str:
    # バイト列をテキストに変換
    # マージされたバイト列は変換できないこともあるため、フォールバックを使用
    s = t.decode('utf-8', errors='replace')

    # 制御文字をエスケープ
    s = replace_control_characters(s)
    return s

render_token(b'Hello\nWorld\t!')

'Hello\\u000aWorld\\u0009!'

In [24]:
class Tokenizer:

    def __init__(self):
        # default: vocab size of 256 (all bytes), no merges, no patterns

        # マージルールの辞書
        # 例: {(65, 66): 256}は、A(65)とB(66)が隣り合っていたらAB(256)にする
        self.merges = {} # (int, int) -> int

        # RegexTokenizerで使用する正規表現パターン
        self.pattern = ""

        # 特殊トークンの辞書
        # 例: {'<|endoftext|>': 100257}
        self.special_tokens = {}

        # 語彙の辞書（ID -> バイト列）
        self.vocab = self._build_vocab()

    def train(self, text, vocab_size, verbose=False):
        raise NotImplementedError

    def encode(self, text):
        raise NotImplementedError

    def decode(self, ids):
        raise NotImplementedError

    def _build_vocab(self):
        """
        self.mergeとself.special_tokensからself.vocabを構築する内部メソッド
        vocabは、IDからバイト列への辞書
        """
        # 初期の256バイトを語彙に追加
        vocab = {idx: bytes([idx]) for idx in range(256)}

        # マージルールを使用して、マージしたバイト列を辞書に追加
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]

        # 特殊トークンを辞書に追加
        for special, idx in self.special_tokens.items():
            vocab[idx] = special.encode("utf-8")

        return vocab

    def save(self, file_prefix):
        """
        2種類のファイルを保存:
        1. .model: トークナイザーを復元するために使用で、正規表現・特殊トークン・マージルールを含む
        2. .vocab: 人間が確認するための可読形式
        """

        # 1. modelファイルを作成

        model_file = file_prefix + ".model"

        with open(model_file, 'w') as f:
            # トークナイザーのバージョン
            f.write("minbpe v1\n")
            # 正規表現
            f.write(f"{self.pattern}\n")
            # 特殊トークンの数
            f.write(f"{len(self.special_tokens)}\n")
            # すべての特殊トークン
            for special, idx in self.special_tokens.items():
                f.write(f"{special} {idx}\n")
            # すべてのマージルール
            for idx1, idx2 in self.merges:
                f.write(f"{idx1} {idx2}\n")

        # 2. vocabファイルを作成

        vocab_file = file_prefix + ".vocab"

        # マージルールの逆引き辞書を作成（マージ後のID -> マージ前のペア）
        inverted_merges = {idx: pair for pair, idx in self.merges.items()}

        with open(vocab_file, "w", encoding="utf-8") as f:
            for idx, token in self.vocab.items():
                # バイト列をテキストに変換
                s = render_token(token)

                # マージ前のトークンがある場合
                if idx in inverted_merges:
                    idx0, idx1 = inverted_merges[idx]
                    s0 = render_token(self.vocab[idx0])
                    s1 = render_token(self.vocab[idx1])
                    # マージ前のトークンも表示
                    f.write(f"[{s0}][{s1}] -> [{s}] {idx}\n")
                else:
                    # そのまま表示
                    f.write(f"[{s}] {idx}\n")

    def load(self, model_file):
        """
        .modelファイルからトークナイザーを復元するメソッド
        """
        assert model_file.endswith(".model")

        merges = {}
        special_tokens = {}
        idx = 256

        with open(model_file, 'r', encoding="utf-8") as f:
            # バージョン
            version = f.readline().strip()
            assert version == "minbpe v1"

            # 正規表現
            self.pattern = f.readline().strip()

            # 特殊トークンの数
            num_special = int(f.readline().strip())

            # 特殊トークンを読み込み
            for _ in range(num_special):
                special, special_idx = f.readline().strip().split()
                special_tokens[special] = int(special_idx)

            # マージルールを読み込み
            for line in f:
                idx1, idx2 = map(int, line.split())
                merges[(idx1, idx2)] = idx
                idx += 1

        # プロパティを更新
        self.merges = merges
        self.special_tokens = special_tokens
        self.vocab = self._build_vocab()

## BasicTokenizer

BasicTokenizerは、GPT-2を参考にした最もシンプルなBPEトークナイザークラス

事前トークン化と特殊トークンは扱っていない

In [25]:
class BasicTokenizer(Tokenizer):

    def __init__(self):
        super().__init__()

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        # 文字列をバイト列に変換
        text_bytes = text.encode("utf-8")

        # 10進数のリストに変換（0から255の整数）
        ids = list(text_bytes)

        # マージの辞書を初期化
        merges = {} # (int, int) -> int

        # 語彙を初期化（int -> bytes）
        vocab = {idx: bytes([idx]) for idx in range(256)}

        # num_mergesまでマージを繰り返す
        for i in range(num_merges):

            # すべてのペアの出現頻度をカウント
            stats = get_stats(ids)

            # 最もカウント数の多いペアを見つける
            pair = max(stats, key=stats.get)

            # 新しいトークンIDを発行
            idx = 256 + i

            # tokensに含まれるペアを新しいIDで置き換える
            ids = merge(ids, pair, idx)

            # 辞書に追加
            merges[pair] = idx

            # 語彙に追加
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

            # 進捗を出力
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")

        # マージルールと語彙をプロパティに保存
        self.merges = merges
        self.vocab = vocab

    def encode(self, text):
        # テキストをUTF-8でバイト列に変換
        text_bytes = text.encode("utf-8")

        # 10進数のリストに変換
        ids = list(text_bytes)

        # マージできなくなるまで繰り返す
        while len(ids) >= 2:

            # 連続するペアの出現回数をカウント
            stats = get_stats(ids)

            # ペアの中から、マージインデックスが最小のものを選ぶ（最も早くマージされたペア）
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))

            # マージインデックスに存在しないペアが終了条件
            if pair not in self.merges:
                break # これ以上マージできない

            # マージし、idsを更新
            idx = self.merges[pair]
            ids = merge(ids, pair, idx)
        return ids

    def decode(self, ids):
        # IDをバイト列に変換
        text_bytes = b"".join(self.vocab[idx] for idx in ids)

        # バイト列をUTF-8でテキストに変換
        text = text_bytes.decode("utf-8", errors="replace")
        return text

# 訓練
basic_tokenizer = BasicTokenizer()
basic_tokenizer.train(long_text, vocab_size=276, verbose=True)

merge 1/20: (227, 129) -> 256 (b'\xe3\x81') had 172168 occurrences
merge 2/20: (227, 130) -> 257 (b'\xe3\x82') had 51415 occurrences
merge 3/20: (227, 128) -> 258 (b'\xe3\x80') had 21639 occurrences
merge 4/20: (256, 174) -> 259 (b'\xe3\x81\xae') had 12791 occurrences
merge 5/20: (256, 132) -> 260 (b'\xe3\x81\x84') had 12067 occurrences
merge 6/20: (256, 170) -> 261 (b'\xe3\x81\xaa') had 9014 occurrences
merge 7/20: (256, 166) -> 262 (b'\xe3\x81\xa6') had 8885 occurrences
merge 8/20: (257, 139) -> 263 (b'\xe3\x82\x8b') had 8844 occurrences
merge 9/20: (256, 168) -> 264 (b'\xe3\x81\xa8') had 8747 occurrences
merge 10/20: (256, 139) -> 265 (b'\xe3\x81\x8b') had 8537 occurrences
merge 11/20: (256, 151) -> 266 (b'\xe3\x81\x97') had 7922 occurrences
merge 12/20: (256, 171) -> 267 (b'\xe3\x81\xab') had 7843 occurrences
merge 13/20: (258, 130) -> 268 (b'\xe3\x80\x82') had 7487 occurrences
merge 14/20: (256, 134) -> 269 (b'\xe3\x81\x86') had 7357 occurrences
merge 15/20: (256, 175) -> 270 (b'\

In [26]:
token = basic_tokenizer.encode("吾輩は猫である。")
len(token), token

(15,
 [229, 144, 190, 232, 188, 169, 270, 231, 140, 171, 274, 256, 130, 263, 268])

In [27]:
basic_tokenizer.decode(token)

'吾輩は猫である。'

In [28]:
basic_tokenizer.save("basic_tokenizer")

# basic_tokenizer.modelは、読み込み用のモデルファイル
# basic_tokenizer.vocabは、人間が読んで確認するための語彙ファイル

In [29]:
basic_tokenizer.load("basic_tokenizer.model")

# 読み込み

## RegexTokenizer

RegexTokenizerは、事前トークン化と特殊トークン（special tokens）を扱う一般的なトークナイザー

事前トークン化により、文字カテゴリの境界を超えてマージされないように設計されている

GPT-2・GPT-4でも導入されている

特殊トークンは、文の区切りや会話の構造タグを示すトークンで、ファインチューニング時に主に追加される

In [30]:
import regex as re

# 事前トークン化のための正規表現
GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

# Tokenizerクラスを継承
class RegexTokenizer(Tokenizer):

    def __init__(self, pattern=None):
        super().__init__()

        # 事前トークン化の正規表現パターンを設定
        self.pattern = GPT4_SPLIT_PATTERN if pattern is None else pattern

        # 正規表現をコンパイル
        self.compiled_pattern = re.compile(self.pattern)

        # 特殊トークンの辞書（str -> int）
        self.special_tokens = {}

        # 特殊トークンの逆引き辞書（int -> str）
        self.inverse_special_tokens = {}

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        # 正規表現でテキストをチャンクに分割
        text_chunks = re.findall(self.compiled_pattern, text)

        # チャンクをバイト列に変換し、10進数のリストに変換
        ids = [list(ch.encode("utf-8")) for ch in text_chunks]

        # マージルールの初期化
        merges = {} # (int, int) -> int

        # 語彙の初期化
        vocab = {idx: bytes([idx]) for idx in range(256)} # idx -> bytes

        for i in range(num_merges):

            # 連続するペアの出現回数をチャンクをまたいでカウント
            stats = {}
            for chunk_ids in ids:
                # チャンクごとにペアの出現数をカウントし、statsを更新
                get_stats(chunk_ids, stats)

            # 最もカウント数の多いペアを見つける
            pair = max(stats, key=stats.get)

            # 新しいトークンIDを発行
            idx = 256 + i

            # チャンクごとにペアを新しいIDで置き換える
            ids = [merge(chunk_ids, pair, idx) for chunk_ids in ids]

            # マージルールを更新
            merges[pair] = idx

            # 語彙を更新
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

            # 進捗を出力
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")

        # プロパティにマージルールと語彙を保存
        self.merges = merges
        self.vocab = vocab

    def register_special_tokens(self, special_tokens):
        """
        特殊トークンを登録するメソッド
        特殊トークンはエンコード・デコード時に使用する
        例: {"<|endoftext|>": 100257}
        """
        # プロパティに保存
        self.special_tokens = special_tokens

        # 逆引き辞書も作成して保存
        self.inverse_special_tokens = {v: k for k, v in special_tokens.items()}

    def _encode_chunk(self, text_bytes):
        """
        テキストチャンクのバイト列を受け取り、トークンIDのリストを返す内部メソッド
        """

        # バイト列を10進数のリストに変換
        ids = list(text_bytes)

        # マージできなくなるまで繰り返す
        while len(ids) >= 2:

            # 連続するペアの出現回数をカウント
            stats = get_stats(ids)

            # ペアの中から、マージインデックスが最小のものを選ぶ（最も早くマージされたペア）
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))

            # マージできなくなったら終了
            if pair not in self.merges:
                break

            # マージし、トークンIDのリストを更新
            idx = self.merges[pair]
            ids = merge(ids, pair, idx)
        return ids

    def encode_ordinary(self, text):
        """
        特殊トークンを考慮しない場合のエンコードメソッド
        """

        # 事前トークン化し、テキストをチャンクに分割
        text_chunks = re.findall(self.compiled_pattern, text)

        # チャンクごとにエンコードし、結果を結合
        ids = []
        for chunk in text_chunks:
            # テキストチャンクをバイト列に変換
            chunk_bytes = chunk.encode("utf-8")

            # バイト列をトークンIDに変換
            chunk_ids = self._encode_chunk(chunk_bytes)

            # 結果を結合
            ids.extend(chunk_ids)
        return ids

    def encode(self, text, allowed_special="none_raise", verbose=False):
        """
        特殊トークンを考慮する場合のエンコードメソッド

        allowed_special: 特殊トークンの扱いを指定
        - "all": すべての特殊トークンを許可
        - "none": 特殊トークンを無視
        - "none_raise": 特殊トークンが含まれている場合はエラーを発生させる
        - set型: 指定された特殊トークンのみを許可
        """

        # decode the user desire w.r.t. handling of special tokens

        # 1. 特殊トークンの扱いを決定
        special = None

        # allの場合
        if allowed_special == "all":
            # すべての特殊トークンを許可
            special = self.special_tokens

        # noneの場合
        elif allowed_special == "none":
            # 特殊トークンを無視
            special = {}

        # none_raiseの場合
        elif allowed_special == "none_raise":
            special = {}
            # 特殊トークンが含まれている場合はエラーを発生させる
            assert all(token not in text for token in self.special_tokens)

        # 特殊トークンがセットで与えられた場合
        elif isinstance(allowed_special, set):
            # 指定された特殊トークンのみを許可
            special = {k: v for k, v in self.special_tokens.items() if k in allowed_special}

        # その他は例外
        else:
            raise ValueError(f"allowed_special={allowed_special} not understood")

        # 特殊トークンがない場合は、普通にエンコード
        if not special:
            return self.encode_ordinary(text)

        # 特殊トークン用の正規表現パターンを作成
        special_pattern = "(" + "|".join(re.escape(k) for k in special) + ")"

        if verbose:
            print(f"special token pattern: {special_pattern}")

        # 特殊トークン用の正規表現でテキストをチャンクに分割
        special_chunks = re.split(special_pattern, text)

        ids = []
        for part in special_chunks:
            # チャンクが特殊トークンの場合
            if part in special:
                # 特殊トークンのIDを追加
                ids.append(special[part])
            else:
                # チャンクを普通にエンコード
                ids.extend(self.encode_ordinary(part))
        return ids

    def decode(self, ids):

        # バッファを初期化
        part_bytes = []

        for idx in ids:
            # IDが語彙に含まれる場合
            if idx in self.vocab:
                # 語彙からバイト列を取得し、バッファに追加
                part_bytes.append(self.vocab[idx])

            # IDが特殊トークンに含まれる場合
            elif idx in self.inverse_special_tokens:
                # 特殊トークンの逆引き辞書からバイト列を取得し、バッファに追加
                part_bytes.append(self.inverse_special_tokens[idx].encode("utf-8"))
            else:
                raise ValueError(f"invalid token id: {idx}")

        # バイト列を結合
        text_bytes = b"".join(part_bytes)

        # バイト列を文字列に変換
        text = text_bytes.decode("utf-8", errors="replace")
        return text

regex_tokenizer = RegexTokenizer()
regex_tokenizer.train(long_text, vocab_size=276, verbose=True)

merge 1/20: (227, 129) -> 256 (b'\xe3\x81') had 172168 occurrences
merge 2/20: (227, 130) -> 257 (b'\xe3\x82') had 51415 occurrences
merge 3/20: (227, 128) -> 258 (b'\xe3\x80') had 21639 occurrences
merge 4/20: (256, 174) -> 259 (b'\xe3\x81\xae') had 12791 occurrences
merge 5/20: (256, 132) -> 260 (b'\xe3\x81\x84') had 12067 occurrences
merge 6/20: (256, 170) -> 261 (b'\xe3\x81\xaa') had 9014 occurrences
merge 7/20: (256, 166) -> 262 (b'\xe3\x81\xa6') had 8885 occurrences
merge 8/20: (257, 139) -> 263 (b'\xe3\x82\x8b') had 8844 occurrences
merge 9/20: (256, 168) -> 264 (b'\xe3\x81\xa8') had 8747 occurrences
merge 10/20: (256, 139) -> 265 (b'\xe3\x81\x8b') had 8537 occurrences
merge 11/20: (256, 151) -> 266 (b'\xe3\x81\x97') had 7922 occurrences
merge 12/20: (256, 171) -> 267 (b'\xe3\x81\xab') had 7843 occurrences
merge 13/20: (258, 130) -> 268 (b'\xe3\x80\x82') had 7487 occurrences
merge 14/20: (256, 134) -> 269 (b'\xe3\x81\x86') had 7357 occurrences
merge 15/20: (256, 175) -> 270 (b'\

In [31]:
token = regex_tokenizer.encode("吾輩は猫である。")
len(token), token

(15,
 [229, 144, 190, 232, 188, 169, 270, 231, 140, 171, 274, 256, 130, 263, 268])

In [32]:
regex_tokenizer.decode(token)

'吾輩は猫である。'

In [33]:
regex_tokenizer.save("regex_tokenizer")

In [34]:
regex_tokenizer.load("regex_tokenizer.model")

## GPT4Tokenizer

GPT4Tokenizerは、BPEの前にバイトシャッフルを適用するGPT-4のトークナイザー

バイトシャッフルは、バイト値を特定の順番

訓練済みのマージルールをOpenAIのサーバーからダウンロードし、Tokenizerクラス用に変換し、再現してみる

In [35]:
# 訓練済みのマージルールのダウンロード

# GPT-4のマージルールをダウンロード
enc = tiktoken.get_encoding("cl100k_base")
enc.name, enc.max_token_value

('cl100k_base', 100276)

In [36]:
# 事前トークン化の正規表現パターン
enc._pat_str

"'(?i:[sdmt]|ll|ve|re)|[^\\r\\n\\p{L}\\p{N}]?+\\p{L}++|\\p{N}{1,3}+| ?[^\\s\\p{L}\\p{N}]++[\\r\\n]*+|\\s++$|\\s*[\\r\\n]|\\s+(?!\\S)|\\s"

In [37]:
# マージルールの辞書（バイト列 -> トークンID）
# tiktokenではトークンIDは、マージの優先順位ランクを示す
# 値が小さいほど優先的にマージされる
enc._mergeable_ranks

import random
rand_index = random.randint(0, enc.max_token_value - 10)
list(enc._mergeable_ranks.items())[rand_index:(rand_index + 10)]

[(b' Rewards', 50868),
 (b' Hog', 50869),
 (b' NSData', 50870),
 (b'stash', 50871),
 (b'Fall', 50872),
 (b' Amer', 50873),
 (b'LinearLayout', 50874),
 (b'/photos', 50875),
 (b' feather', 50876),
 (b' |\r\n', 50877)]

In [38]:
# 特殊トークンの辞書（文字列 -> トークンID）
enc._special_tokens

{'<|endoftext|>': 100257,
 '<|fim_prefix|>': 100258,
 '<|fim_middle|>': 100259,
 '<|fim_suffix|>': 100260,
 '<|endofprompt|>': 100276}

In [39]:
# bpeは、1つのトークンをマージルールに従ってmax_rankまで分解する関数
# tiktoken用マージルールをTokenizerクラス用マージルールに変換するrecover_merge関数で使用する

def bpe(mergeable_ranks, token, max_rank):

    # トークンIDをバイト列に変換し、1バイトずつのリストに分解
    parts = [bytes([b]) for b in token]

    # マージルールに従ってマージ
    while True:
        min_idx = None
        min_rank = None

        # 連続するペアの中で、最もランクが小さいペアを見つける
        for i, pair in enumerate(zip(parts[:-1], parts[1:])):
            rank = mergeable_ranks.get(pair[0] + pair[1])
            if rank is not None and (min_rank is None or rank < min_rank):
                min_idx = i
                min_rank = rank

        # マージできなくなる、または最大ランクに達したら終了
        if min_rank is None or (max_rank is not None and min_rank >= max_rank):
            break

        assert min_idx is not None

        # 最もランクが小さいペアをマージ
        parts = parts[:min_idx] + [parts[min_idx] + parts[min_idx + 1]] + parts[min_idx + 2:]

    return parts

print(bpe(enc._mergeable_ranks, b"HuggingFace", 100))
print(bpe(enc._mergeable_ranks, b"HuggingFace", 1000))
print(bpe(enc._mergeable_ranks, b"HuggingFace", 10000))
print(bpe(enc._mergeable_ranks, b"HuggingFace", None))

[b'H', b'u', b'g', b'g', b'i', b'n', b'g', b'F', b'a', b'c', b'e']
[b'H', b'ug', b'g', b'ing', b'F', b'ace']
[b'H', b'ugg', b'ing', b'F', b'ace']
[b'H', b'ugging', b'Face']


In [40]:
# recover_mergeは、tiktoken用マージルールをTokenizerクラス用マージルールに変換する関数
# tiktoken用マージルール: {(b'abo', 48521), ...}
# Tokenizer用マージルール: {(97, 98): 256, ...}

def recover_merges(mergeable_ranks):
    merges = {}

    # すべてのマージ可能なトークンを処理
    for token, rank in mergeable_ranks.items():

        # 長さが1の場合はスキップ
        if len(token) == 1:
            continue

        # tokenをmax_rank=rankでbpe分解し、ペアを取得
        pair = tuple(bpe(mergeable_ranks, token, max_rank=rank))

        assert len(pair) == 2

        # ペアを復元
        ix0 = mergeable_ranks[pair[0]]
        ix1 = mergeable_ranks[pair[1]]
        merges[(ix0, ix1)] = rank

    return merges

In [41]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

GPT4_SPECIAL_TOKENS = {
    '<|endoftext|>': 100257,
    '<|fim_prefix|>': 100258,
    '<|fim_middle|>': 100259,
    '<|fim_suffix|>': 100260,
    '<|endofprompt|>': 100276
}

# RegexTokenierを継承
class GPT4Tokenizer(RegexTokenizer):

    def __init__(self):
        super().__init__(pattern=GPT4_SPLIT_PATTERN)

        # tiktokenのマージルールをダウンロード
        enc = tiktoken.get_encoding("cl100k_base")
        mergeable_ranks = enc._mergeable_ranks

        # Tokenizerクラス用のマージルールに復元
        self.merges = recover_merges(mergeable_ranks)

        # 語彙を初期化（ID -> バイト列）
        vocab = {idx: bytes([idx]) for idx in range(256)}

        # マージルールから語彙を復元
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]

        # 語彙をプロパティに設定
        self.vocab = vocab

        # 0~255のバイト値をGPT-4のバイトシャッフルに従ってマッピング
        self.byte_shuffle = {i: mergeable_ranks[bytes([i])] for i in range(256)}

        # バイトシャッフルの逆引き辞書も作成
        self.inverse_byte_shuffle = {v: k for k, v in self.byte_shuffle.items()}

        # GPT-4の特殊トークンを登録
        self.register_special_tokens(GPT4_SPECIAL_TOKENS)

    def _encode_chunk(self, text_bytes):
        """
        テキストチャンクのバイト列を受け取り、トークンIDのリストを返す内部メソッド
        """
        # バイトシャッフルでtiktokenの内部的な順番に並び替え
        text_bytes = bytes(self.byte_shuffle[b] for b in text_bytes)
        ids = super()._encode_chunk(text_bytes)
        return ids

    def decode(self, ids):
        # IDをバイト列に変換
        text_bytes = b"".join(self.vocab[idx] for idx in ids)

        # バイトシャッフルの逆引きで元のバイト順に戻す
        text_bytes = bytes(self.inverse_byte_shuffle[b] for b in text_bytes)

        # バイト列を文字列に変換
        text = text_bytes.decode("utf-8", errors="replace")
        return text

    def train(self, text, vocab_size, verbose=False):
        raise NotImplementedError

    def save(self, file_prefix):
        raise NotImplementedError("GPT4Tokenizer cannot be saved.")

    def load(self, model_file):
        raise NotImplementedError("GPT4Tokenizer cannot be loaded.")

    def save_vocab(self, vocab_file):
        """
        人が確認するための語彙ファイルを保存するメソッド
        形式はBaseTokenizerと同じ
        """
        # 初期の256バイトをシャッフルして語彙に追加
        vocab = {idx: bytes([self.inverse_byte_shuffle[idx]]) for idx in range(256)}

        # マージルールを使用して、マージしたバイト列を辞書に追加
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]

        # マージルールの逆引き辞書を作成（マージ後のID -> マージ前のペア）
        inverted_merges = {idx: pair for pair, idx in self.merges.items()}

        # 語彙ファイルを作成
        with open(vocab_file, "w", encoding="utf-8") as f:
            # すべてのトークンをID順に出力
            for idx, token in vocab.items():
                # バイト列を人が読める形式に変換
                s = render_token(token)

                # マージ前のトークンがある場合
                if idx in inverted_merges:
                    # 分解して出力
                    idx0, idx1 = inverted_merges[idx]
                    s0 = render_token(vocab[idx0])
                    s1 = render_token(vocab[idx1])
                    f.write(f"[{s0}][{s1}] -> [{s}] {idx}\n")
                else:
                    # そのまま出力
                    f.write(f"[{s}] {idx}\n")

gpt4_tokenizer = GPT4Tokenizer()
gpt4_tokenizer.save_vocab("gpt4.vocab")

In [42]:
# GPT-4トークナイザーと同じ結果が得られる

text = "hello123!!!? (안녕하세요!) 😉"
enc = tiktoken.get_encoding("cl100k_base")
tokenizer = GPT4Tokenizer()

print("tiktoken", enc.encode(text))
print("GPT4Tokenizer", tokenizer.encode(text))

tiktoken [15339, 4513, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 57037]
GPT4Tokenizer [15339, 4513, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 57037]


In [43]:
# 特殊トークンも同様

text = "<|endoftext|>hello world"
enc = tiktoken.get_encoding("cl100k_base")
tokenizer = GPT4Tokenizer()

print("tiktoken", enc.encode(text, allowed_special="all"))
print("GPT4Tokenizer", tokenizer.encode(text, allowed_special="all"))

tiktoken [100257, 15339, 1917]
GPT4Tokenizer [100257, 15339, 1917]


## Rust

In [48]:
!maturin develop --release --manifest-path nanochat/rustbpe/Cargo.toml

    Updating crates.io index
  Downloaded ahash v0.8.12                                                 
  Downloaded itoa v1.0.15                                                  
  Downloaded autocfg v1.5.0ing bytes: 110.7KiB                             
  Downloaded heck v0.5.0aining bytes: 74.2KiB                              
  Downloaded equivalent v1.0.2 bytes: 1.4MiB                               
  Downloaded crossbeam-deque v0.8.6s: 1.4MiB                               
  Downloaded either v1.15.0ing bytes: 5.1MiB                               
  Downloaded memoffset v0.9.1g bytes: 5.1MiB                               
  Downloaded indoc v2.0.6ining bytes: 5.0MiB                               
  Downloaded bit-set v0.8.0ing bytes: 5.0MiB                               
  Downloaded unindent v0.2.4ng bytes: 5.0MiB                               
  Downloaded version_check v0.9.5tes: 4.9MiB                               
  Downloaded pyo3-macros v0.23.5ytes: 4.8MiB               

In [ ]:
import rustbpe

rustbpe_tokenizer = rustbpe.Tokenizer()
rustbpe_tokenizer.train_from_iterator([long_text], vocab_size=276)

In [54]:
rustbpe_tokenizer.encode("吾輩は猫である。")

[229, 144, 190, 232, 188, 169, 270, 231, 140, 171, 274, 256, 130, 263, 268]

In [55]:
rustbpe_tokenizer.get_pattern()

"'(?i:[sdmt]|ll|ve|re)|[^\\r\\n\\p{L}\\p{N}]?+\\p{L}+|\\p{N}{1,3}| ?[^\\s\\p{L}\\p{N}]++[\\r\\n]*|\\s*[\\r\\n]|\\s+(?!\\S)|\\s+"

## ベンチマーク

英語版のWikipediaデータ[enwiki8][1]を使用して検証

- UTF-8エンコードされたXML
- enwiki8.zipは、36MB
- enwiki9.zipは、323MB

[1]: https://mattmahoney.net/dc/textdata.html

In [ ]:
def enwik8_path():
    """Fixture to download and cache enwik8 dataset."""
    home_dir = os.path.expanduser("~")
    base_dir = os.path.join(home_dir, ".cache", "bpe")
    os.makedirs(base_dir, exist_ok=True)
    enwik8_url = "https://mattmahoney.net/dc/enwik8.zip"
    enwik8_local_path = os.path.join(base_dir, "enwik8")
    enwik8_local_path_zip = os.path.join(base_dir, "enwik8.zip")
    if not os.path.exists(enwik8_local_path):
        print(f"Downloading enwik8 to {enwik8_local_path_zip}")
        import requests
        response = requests.get(enwik8_url)
        with open(enwik8_local_path_zip, "wb") as f:
            f.write(response.content)
        with zipfile.ZipFile(enwik8_local_path_zip, "r") as zip_ref:
            zip_ref.extractall(base_dir)
        print(f"Unzipped enwik8 to {enwik8_local_path}")
        os.remove(enwik8_local_path_zip)
        print(f"Removed {enwik8_local_path_zip}")
    else:
        print(f"Using existing enwik8 at {enwik8_local_path}")
    return enwik8_local_path

enwik8_path()

Using existing enwik8 at /root/.cache/bpe/enwik8


'/root/.cache/bpe/enwik8'

In [67]:
def enwik8_small(enwik8_path):
    """100KBのenwik8を提供する関数"""
    with open(enwik8_path, "r", encoding="utf-8") as f:
        return f.read(100_000)

In [68]:
def enwik8_large(enwik8_path):
    """10MBのenwik8を提供する関数"""
    with open(enwik8_path, "r", encoding="utf-8") as f:
        return f.read(10**7)

In [ ]:
def time_function(func, *args, **kwargs):
    """
    関数funcの実行時間を測定するユーティリティ関数
    """
    start_time = time.time()
    result = func(*args, **kwargs)
    end_time = time.time()
    elapsed = end_time - start_time
    return result, elapsed

enwik8_smallでの訓練時間を比較

In [73]:
regex_tokenizer = RegexTokenizer()
time_function(regex_tokenizer.train, enwik8_small(enwik8_path()), vocab_size=2048)

Using existing enwik8 at /root/.cache/bpe/enwik8


(None, 17.963773012161255)

In [76]:
rustbpe_tokenizer = rustbpe.Tokenizer()
time_function(rustbpe_tokenizer.train_from_iterator, [enwik8_small(enwik8_path())], vocab_size=2048)

Using existing enwik8 at /root/.cache/bpe/enwik8


(None, 0.015126943588256836)

enwik8_largeでのエンコード時間を比較

In [84]:
regex_result = time_function(regex_tokenizer.encode, enwik8_large(enwik8_path()))
len(regex_result[0]), regex_result[1]

Using existing enwik8 at /root/.cache/bpe/enwik8


(4191455, 7.271657228469849)

In [83]:
rustbpe_result = time_function(rustbpe_tokenizer.encode, enwik8_large(enwik8_path()))
len(rustbpe_result[0]), rustbpe_result[1]

Using existing enwik8 at /root/.cache/bpe/enwik8


(30129, 0.0066034793853759766)